# CS7002NU Assessment 1 — PPE Detection System
**London Building Materials Manufacturing Ltd.**

Computer vision system to detect PPE compliance (helmets, hi-vis vests, goggles) from factory camera feeds.

| Layer | Tool |
|-------|------|
| Code | GitHub (`lmu-ai-vision-dl-assessment`) |
| Compute | Google Colab T4 GPU |
| Storage | Google Drive (`MyDrive/CS7002NU_PPE/`) |

## 0. Environment Setup
Mount Google Drive, clone the GitHub repo, install dependencies, add `src/` to path.

> **Run this cell first on every new Colab session.**

In [ ]:
import os, sys

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/CS7002NU_PPE'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'

for d in [f'{OUTPUT_DIR}/section_a', f'{OUTPUT_DIR}/section_b']:
    os.makedirs(d, exist_ok=True)

# 2. Clone / pull GitHub repo
REPO_URL = 'https://github.com/hishamsharif/lmu-ai-vision-dl-assessment.git'
REPO_DIR = '/content/lmu-ai-vision-dl-assessment'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q opencv-python-headless

# 4. Verify
import cv2, numpy as np, matplotlib
print('cv2     :', cv2.__version__)
print('numpy   :', np.__version__)
print('Setup complete')

---
## Section A: Image Processing

### A.1 Image Blending
Blend Image A and Image B using three alpha weight ratios: **0.60/0.40**, **0.75/0.25**, **0.50/0.50**.

Formula: `output(x,y) = α · imgA(x,y) + (1 − α) · imgB(x,y)`

Implementation: `cv2.addWeighted(imgA, α, imgB, 1−α, gamma=0)`

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.section_a.image_ops import blend

# Load images (upload from local Datasets/ folder to Drive first)
img_a = cv2.imread(f'{DRIVE_ROOT}/datasets/raw/Image A.jpeg')
img_b = cv2.imread(f'{DRIVE_ROOT}/datasets/raw/Image B.jpeg')
assert img_a is not None, 'Image A not found — upload to Drive/datasets/raw/'
assert img_b is not None, 'Image B not found — upload to Drive/datasets/raw/'
print(f'Image A: {img_a.shape}   Image B: {img_b.shape}')

In [ ]:
alphas = [0.60, 0.75, 0.50]
labels = ['60% A / 40% B', '75% A / 25% B', '50% A / 50% B']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('A.1 — Image Blending Results', fontsize=14, fontweight='bold')

for ax, alpha, label in zip(axes, alphas, labels):
    result = blend(img_a, img_b, alpha)
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=11)
    ax.axis('off')

plt.tight_layout()
out = f'{OUTPUT_DIR}/section_a/blending_results.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out)

### A.2 Histogram Equalisation
Apply histogram equalisation to Image C to improve contrast under uneven factory floor lighting.

Algorithm: remap pixel intensities via the cumulative distribution function (CDF) to span the full 0–255 range.

Implementation: `cv2.equalizeHist()` on grayscale channel.

In [ ]:
from src.section_a.image_ops import equalize_histogram

img_c = cv2.imread(f'{DRIVE_ROOT}/datasets/raw/Image C.jpeg')
assert img_c is not None, 'Image C not found — upload to Drive/datasets/raw/'

img_c_eq = equalize_histogram(img_c)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('A.2 — Histogram Equalisation', fontsize=14, fontweight='bold')

axes[0, 0].imshow(cv2.cvtColor(img_c, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original (Image C)'); axes[0, 0].axis('off')

axes[0, 1].imshow(cv2.cvtColor(img_c_eq, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('After Histogram Equalisation'); axes[0, 1].axis('off')

gray_orig = cv2.cvtColor(img_c, cv2.COLOR_BGR2GRAY)
gray_eq   = cv2.cvtColor(img_c_eq, cv2.COLOR_BGR2GRAY)

axes[1, 0].hist(gray_orig.ravel(), 256, [0, 256], color='steelblue')
axes[1, 0].set_title('Histogram — Original'); axes[1, 0].set_xlabel('Pixel intensity')

axes[1, 1].hist(gray_eq.ravel(), 256, [0, 256], color='darkorange')
axes[1, 1].set_title('Histogram — Equalised'); axes[1, 1].set_xlabel('Pixel intensity')

plt.tight_layout()
out = f'{OUTPUT_DIR}/section_a/histogram_equalisation.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out)

### A.3 Image Processing Techniques
Apply brightness adjustment, horizontal flip, vertical flip, and 90° clockwise rotation to Image C.

Each operation links to a practical data augmentation use case for factory camera feeds:
- **Brightness** → variable lighting conditions / time of day
- **Horizontal flip** → worker facing either direction
- **Vertical flip** → inverted or ceiling-mounted camera
- **Rotation** → tilted camera angle

In [ ]:
from src.section_a.image_ops import adjust_brightness, flip_image, rotate_90cw

transforms = [
    (img_c,                              'Original'),
    (adjust_brightness(img_c, beta=50),  'Brightness +50'),
    (flip_image(img_c, 'horizontal'),    'Horizontal Flip'),
    (flip_image(img_c, 'vertical'),      'Vertical Flip'),
    (rotate_90cw(img_c),                 '90° Clockwise'),
]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('A.3 — Image Processing Techniques (Image C)', fontsize=14, fontweight='bold')

for ax, (result, title) in zip(axes, transforms):
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
out = f'{OUTPUT_DIR}/section_a/image_transforms.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out)

---
## Section B: Deep Learning Object Detection

> Section B cells will be added incrementally as Tasks 3–6 are implemented.